## Active Learning experiment

Compares `entropy` vs `random` strategies and computes the rubric metric: `saved_examples` = how many fewer labeled examples entropy needs to reach the random strategy's peak F1.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

runs = sorted(Path('artifacts').glob('run_*/reports/al_history.csv'))
path = runs[-1] if runs else None
print('al_history path:', path)
history = pd.read_csv(path) if path else pd.DataFrame()
history.head()

In [ ]:
if not history.empty:
    plt.figure(figsize=(8,5))
    for strat in history['strategy'].unique():
        sub = history[history['strategy'] == strat].sort_values('n_labeled')
        plt.plot(sub['n_labeled'], sub['f1'], marker='o', label=strat)
    plt.xlabel('n_labeled')
    plt.ylabel('F1 (macro)')
    plt.title('Active Learning: entropy vs random')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    rand = history[history['strategy'] == 'random'].sort_values('n_labeled')
    ent = history[history['strategy'] == 'entropy'].sort_values('n_labeled')
    target_f1 = float(rand['f1'].max())

    required_random = rand[rand['f1'] >= target_f1].sort_values('n_labeled').iloc[0]['n_labeled']
    required_entropy = ent[ent['f1'] >= target_f1].sort_values('n_labeled').iloc[0]['n_labeled']
    saved_examples = int(required_random - required_entropy)
    saved_examples
else:
    'No al_history.csv found yet. Run `python run_pipeline.py` first.'